In [2]:
from torch.utils.data import DataLoader, Dataset
from torch import nn
import torch
import os
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np

def calculate_metrics(y_pred, y_true, threshold=0.5, epsilon=1e-7):

    probs=torch.sigmoid(y_pred)

    y_pred=(probs>threshold).float()
    y_true=y_true.float()

    y_pred=y_pred.view(-1)
    y_true=y_true.view(-1)

    intersection = (y_pred * y_true).sum()
    total = (y_pred.sum() + y_true.sum())
    union = total - intersection

    iou = (intersection + epsilon) / (union + epsilon)
    dice = (2 * intersection + epsilon) / (total + epsilon)

    return iou.item(), dice.item()

class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, inputs, targets, epsilon=1e-7):

        inputs=torch.sigmoid(inputs)

        inputs=inputs.view(-1)
        targets=targets.view(-1)

        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2 * intersection + epsilon) / (inputs.sum() + targets.sum() + epsilon)
        bce=nn.functional.binary_cross_entropy(inputs, targets, reduction="mean")

        return dice_loss + bce

segmentation_transforms=A.Compose([
    A.Resize(256,256),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class SegmentationData(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir=image_dir
        self.mask_dir=mask_dir

        self.images=os.listdir(image_dir)
        self.masks=os.listdir(mask_dir)
        self.transform=transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path=os.path.join(self.image_dir, self.images[idx])
        mask_path=os.path.join(self.mask_dir, self.masks[idx])

        image=Image.open(image_path)

        image_array = np.array(image, dtype = np.float32)

        min_hu = -1000
        max_hu = 400

        image_windowed=np.clip(image_array, min_hu, max_hu)
        image = ((image_windowed - min_hu) / (max_hu - min_hu) * 255).astype(np.uint8)

        mask = Image.open(mask_path)
        mask_array = np.array(mask)
        mask = (mask_array > 0).astype(np.float32)

        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image=augmented["image"]
            mask=augmented["mask"]

        mask=torch.unsqueeze(mask, 0)

        return image, mask

class UNET(nn.Module):
    def __init__(self):
        super().__init__()

        self.convblock1=nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1, 1),
            nn.ReLU()
        )

        self.convblock2=nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, 1, 1),
            nn.ReLU()
        )

        self.convblock3=nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.ReLU()
        )

        self.convblock4=nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, 1, 1),
            nn.ReLU()
        )

        self.convblock5=nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(512, 1024, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(1024, 1024, 3, 1, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(1024, 512, 2, 2)
        )

        self.upconv1=nn.Sequential(
            nn.Conv2d(1024, 512, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(512, 512, 3, 1, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(512, 256, 2, 2)
        )

        self.upconv2=nn.Sequential(
            nn.Conv2d(512, 256, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 2, 2)
        )

        self.upconv3=nn.Sequential(
            nn.Conv2d(256, 128, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, 1, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 2, 2)
        )

        self.segmentation_convblock=nn.Sequential(
            nn.Conv2d(128, 64, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(64, 1, 1)
        )

    def copy_tensor(self, contracting_tensor, expansive_tensor):

        return torch.cat([contracting_tensor, expansive_tensor], 1)

    def forward(self, x):
        x=self.convblock1(x)
        copy1=x
        x=self.convblock2(x)
        copy2=x
        x=self.convblock3(x)
        copy3=x
        x=self.convblock4(x)
        copy4=x
        x=self.convblock5(x)
        x=self.copy_tensor(copy4, x)
        x=self.upconv1(x)
        x=self.copy_tensor(copy3, x)
        x=self.upconv2(x)
        x=self.copy_tensor(copy2, x)
        x=self.upconv3(x)
        x=self.copy_tensor(copy1, x)
        x=self.segmentation_convblock(x)

        return x

from torch.utils.data import DataLoader

def create_dataloaders(train_image_dir="split/train/images",
                       train_mask_dir="split/train/masks",
                       val_image_dir="split/val/images",
                       val_mask_dir="split/val/masks",
                       batch_size=8):

    train_data=SegmentationData(train_image_dir, train_mask_dir, transform=segmentation_transforms)
    val_data=SegmentationData(val_image_dir, val_mask_dir, transform=segmentation_transforms)

    train_dataloader=DataLoader(dataset=train_data, batch_size=batch_size, shuffle=True, num_workers=0)
    val_dataloader=DataLoader(dataset=val_data, batch_size=batch_size, shuffle=False, num_workers=0)

    return train_dataloader, val_dataloader

def train_step(model, dataloader, loss_fn, optimizer, device):

    model.train()

    train_loss, iou_score, dice_score = 0, 0, 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)

        y_preds=model(X)

        iou, dice = calculate_metrics(y_preds, y)
        iou_score+=iou
        dice_score+=dice

        loss=loss_fn(y_preds, y)
        train_loss+=loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss=train_loss/len(dataloader)
    iou_score=iou_score/len(dataloader)
    dice_score=dice_score/len(dataloader)

    return train_loss, iou_score, dice_score

def val_step(model, dataloader, loss_fn, device):

    model.eval()

    with torch.inference_mode():

        val_loss, iou_score, dice_score = 0, 0, 0

        for X, y in dataloader:
            X, y = X.to(device), y.to(device)

            y_preds=model(X)

            iou, dice = calculate_metrics(y_preds, y)
            iou_score+=iou
            dice_score+=dice

            loss=loss_fn(y_preds, y)
            val_loss+=loss.item()


    val_loss=val_loss/len(dataloader)
    iou_score=iou_score/len(dataloader)
    dice_score=dice_score/len(dataloader)

    return val_loss, iou_score, dice_score

def train(model, train_dataloader, val_dataloader, loss_fn, optimizer, device, epochs=10):
    train_loss_list=[]
    train_iou_list=[]
    train_dice_list=[]

    val_loss_list=[]
    val_iou_list=[]
    val_dice_list=[]

    model.to(device)

    for epoch in range(epochs):
        train_loss, train_iou, train_dice = train_step(model=model, dataloader=train_dataloader, loss_fn=loss_fn, optimizer=optimizer, device=device)
        val_loss, val_iou, val_dice = val_step(model=model, dataloader=val_dataloader, loss_fn=loss_fn, device=device)

        print(f"Epochs: {epoch+1}, Train Loss: {train_loss:.4f}, Train IoU: {train_iou:.4f}, Train DICE: {train_dice:.4f}, Val loss: {val_loss:.4f}, Val IoU: {val_iou:.4f}, Val DICE: {val_dice:.4f}")
        train_loss_list.append(train_loss)
        train_iou_list.append(train_iou)
        train_dice_list.append(train_dice)

        val_loss_list.append(val_loss)
        val_iou_list.append(val_iou)
        val_dice_list.append(val_dice)

    return {"Train Loss": train_loss_list, "Train IoU": train_iou_list, "Train DICE": train_dice_list, "Val Loss": val_loss_list, "Val IoU": val_iou_list, "Val DICE": val_dice_list}

model=UNET()

loss_fn=DiceBCELoss()
optimizer=torch.optim.Adam(model.parameters(), lr=0.001)

train_dataloader, val_dataloader = create_dataloaders(batch_size=1)

results=train(model=model, train_dataloader=train_dataloader, val_dataloader=val_dataloader, loss_fn=loss_fn, optimizer=optimizer, device="cuda", epochs=100)

torch.save(model.state_dict(), "epochs100.pt")

Epochs: 1, Train Loss: 0.7778, Train IoU: 0.5968, Train DICE: 0.6961, Val loss: 0.3532, Val IoU: 0.7843, Val DICE: 0.8710
Epochs: 2, Train Loss: 0.4857, Train IoU: 0.7334, Train DICE: 0.8213, Val loss: 0.1571, Val IoU: 0.9026, Val DICE: 0.9451
Epochs: 3, Train Loss: 0.2492, Train IoU: 0.8832, Train DICE: 0.9310, Val loss: 0.2394, Val IoU: 0.8600, Val DICE: 0.9190
Epochs: 4, Train Loss: 0.3426, Train IoU: 0.8521, Train DICE: 0.9083, Val loss: 0.0905, Val IoU: 0.9342, Val DICE: 0.9648
Epochs: 5, Train Loss: 0.1880, Train IoU: 0.9080, Train DICE: 0.9478, Val loss: 0.0892, Val IoU: 0.9433, Val DICE: 0.9704
Epochs: 6, Train Loss: 0.2383, Train IoU: 0.9132, Train DICE: 0.9508, Val loss: 0.0944, Val IoU: 0.9304, Val DICE: 0.9632
Epochs: 7, Train Loss: 0.1915, Train IoU: 0.9085, Train DICE: 0.9486, Val loss: 0.1215, Val IoU: 0.9350, Val DICE: 0.9659
Epochs: 8, Train Loss: 0.1826, Train IoU: 0.9290, Train DICE: 0.9600, Val loss: 0.0697, Val IoU: 0.9539, Val DICE: 0.9759
Epochs: 9, Train Loss: 0